## Preliminaries
### Install DICES with spaCy extensions
 - Models have to be manually installed
 - **You must restart kernel afterwards**
 - Don't re-run this cell after restart — just carry on below

In [2]:
!pip install -qqq ../'dices-client[spacy]'
!pip install -qqq https://huggingface.co/latincy/la_core_web_trf/resolve/main/la_core_web_trf-3.9.3-py3-none-any.whl
!pip install -qqq https://huggingface.co/chcaa/grc_odycy_joint_trf/resolve/main/grc_odycy_joint_trf-0.7.0-py3-none-any.whl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grc-odycy-joint-trf 0.7.0 requires spacy<3.8.0,>=3.7.4, but you have spacy 3.8.14 which is incompatible.
ERROR: Wheel 'grc-odycy-joint-trf' located at /private/var/folders/2x/rfhzgyjn2zq2wlzk_kn4th680000gn/T/pip-unpack-iv3571zz/grc_odycy_joint_trf-0.7.0-py3-none-any.whl is invalid.


## Main code begins here

### Import statements

In [11]:
from dicesapi import DicesAPI
import pandas as pd
from tqdm import tqdm

### Initialize new DICES session

If something hasn't been installed correctly, these may throw errors. Warnings about spacy version can be ignored. If you install any new packages, restart the kernel and begin again from the import statements above.

In [12]:
api = DicesAPI()
api.initializeCts()
api.initializeNlp(
    latin_model="la_core_web_trf",
    greek_model="grc_odycy_joint_trf",
)

### Query DICES and get Flavian speeches

In [13]:
speeches = (
    api.getSpeeches(author_name="Silius") +
    api.getSpeeches(author_name="Statius") +
    api.getSpeeches(author_name="Valerius Flaccus")
)

# after addition operator things get shuffled, have to re-sort
speeches.sort()

print(f"got {len(speeches)} speeches")

got 819 speeches


### Download text passages from Perseus

**Note**: This process works slightly differently now. The old `cts.getPassage(speech)` has been replaced with `speech.fetchPassage()`.

In [15]:
for s in tqdm(speeches):
    s.fetchPassage()

100%|████████████████████████████████████████| 819/819 [04:04<00:00,  3.35it/s]


### Parse with spaCy

I've changed the underlying code a bit here, but I think the method call looks the same to the user.

In [16]:
for s in tqdm(speeches):
    s.passage.runSpacyPipeline()

  0%|                                                  | 0/819 [00:00<?, ?it/s]/Users/chris/Documents/git/dices-project-container/dices-examples/venv/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.autocast(device_type="cuda", enabled=self._mixed_precision):
  5%|██                                       | 41/819 [00:20<12:49,  1.01it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors
Token indices

### Build token table

In [28]:
tokens = []

for s in speeches:
    for tok in s.passage.spacy_doc:
        i = s.passage.getLineIndex(tok)

        # get prefix for locus
        pref = s.l_fi.split(".")[0]

        tokens.append(dict(
            author = s.work.author.name,
            work = s.work.title,
            pref = pref,
            seq = s.passage.line_array[i]["seq"],
            line = s.passage.line_array[i]["n"],
            spkr = s.getSpkrString(),
            addr = s.getAddrString(),
            type = s.type,
            tags = ";".join([tag["type"] for tag in s._attributes["tags"]]),
            token = tok.text,
            lemma = tok.lemma_,
            pos = tok.pos_,
            verbform = ";".join(tok.morph.get("VerbForm")),                
            mood = ";".join(tok.morph.get("Mood")),
            tense = ";".join(tok.morph.get("Tense")),                
            voice = ";".join(tok.morph.get("Voice")),
            person = ";".join(tok.morph.get("Person")),
            number = ";".join(tok.morph.get("Number")),
            case = ";".join(tok.morph.get("Case")),
            gender = ";".join(tok.morph.get("Gender")),
        ))

tokens = pd.DataFrame(tokens)
tokens.to_csv("flav_tokens.csv", index=False)
display(tokens)

,author,work,pref,seq,line,spkr,addr,type,tags,token,lemma,pos,verbform,mood,tense,voice,person,number,case,gender
0,Silius,Punica,1,0,42,Juno,"river Aufidus, river Ticinus",S,ora;exh;nar,Intulerit,infero,VERB,Fin,Sub,Past,Act,3,Sing,,
1,Silius,Punica,1,0,42,Juno,"river Aufidus, river Ticinus",S,ora;exh;nar,Latio,Latium,PROPN,,,,,,Sing,Dat,Neut
2,Silius,Punica,1,0,42,Juno,"river Aufidus, river Ticinus",S,ora;exh;nar,",",",",PUNCT,,,,,,,,
3,Silius,Punica,1,0,42,Juno,"river Aufidus, river Ticinus",S,ora;exh;nar,spreta,sperno,VERB,Part,,,Pass,,Plur,Acc,Neut
4,Silius,Punica,1,0,42,Juno,"river Aufidus, river Ticinus",S,ora;exh;nar,me,ego,PRON,,,,,1,Sing,Abl,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84479,Valerius Flaccus,Argonautica,8,0,467,Jason,Medea,D,res;per,uelle,uolo,VERB,Inf,,,Act,,,,
84480,Valerius Flaccus,Argonautica,8,0,467,Jason,Medea,D,res;per,.,.,PUNCT,,,,,,,,
84481,Valerius Flaccus,Argonautica,8,0,467,Jason,Medea,D,res;per,.,.,PUNCT,,,,,,,,
84482,Valerius Flaccus,Argonautica,8,0,467,Jason,Medea,D,res;per,.,.,PUNCT,,,,,,,,
